# Notebook 2 — Ablation Studies
**RSNA Intracranial Hemorrhage Detection — Improved Pipeline**

Controlled experiments to measure the contribution of each improvement.
Each experiment trains on a **15% subset** for **3 epochs** (one fold only)
to keep compute within a single Kaggle session.

> **⚠️ Honest Disclosure**: These ablation results indicate *directional trends*
> under a heavily constrained experimental setting (15% training subset,
> 3 epochs, single fold). They should **not** be interpreted as definitive
> proof of superiority. Full-data, full-epoch ablations would be needed
> for stronger claims. We report these to show that each design choice
> was empirically motivated, not arbitrary.

### Experiments
| # | Variable | Control | Treatment |
|---|----------|---------|----------|
| 1 | **Input** | 2D (3ch) | 2.5D (9ch) |
| 2 | **Backbone** | EfficientNet-B0 | EfficientNet-B4 |
| 3 | **Formulation** | Binary (any only) | Multi-label (6 outputs) |

| 4 | **Augmentation** | Baseline (flip) | Full (MixUp+CutMix+CoarseDropout) |- **Time**: ~2-3h total

| 5 | **Hard neg mining** | OFF | ON (top-500, 3×) |- **Inputs**: NB00 output + NB02 NPY cache + NB01 best model (fold 0)

- **GPU**: Yes (T4 sufficient)
### Requirements

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  CONFIG  ██
# ══════════════════════════════════════════════════════════════════════════
from pathlib import Path
import os, json, gc, time, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import roc_auc_score
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ─── Paths ────────────────────────────────────────────────────────────
NB00_DIR = Path('/kaggle/input/notebooks/harshitghosh/00-preprocess-metadata')
NB02_DIR = Path('/kaggle/input/notebooks/harshitghosh/nb02eda')
NB01_DIR = Path('/kaggle/input/notebooks/harshitghosh/01-training')  # latest session

MANIFEST_PATH  = NB00_DIR / 'manifest_2_5d.csv'
NPY_CACHE_DIR  = NB02_DIR / 'cache'
NORM_STATS     = NB02_DIR / 'normalization_stats.json'

OUTPUT_DIR = Path('/kaggle/working')

ABLATION_FRAC = 0.15   # fraction of train used per experiment
ABLATION_EP   = 3      # epochs per experiment
IMG_SIZE      = 256
BATCH_SIZE    = 16
LR            = 3e-4
SEED          = 42
FOLD_ID       = 0
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']

# Seeds
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

seed_everything(SEED)

# Norm stats
if NORM_STATS.exists():
    _ns = json.load(open(NORM_STATS))
    MEAN_3CH = np.array(_ns['mean'], dtype=np.float32)
    STD_3CH  = np.array(_ns['std'],  dtype=np.float32)
else:
    MEAN_3CH = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD_3CH  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
MEAN_9CH = np.tile(MEAN_3CH, 3)
STD_9CH  = np.tile(STD_3CH, 3)

print(f'Device: {DEVICE}')
print(f'Ablation: {ABLATION_FRAC*100:.0f}% subset, {ABLATION_EP} epochs each')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  DATASETS  ██
# ══════════════════════════════════════════════════════════════════════════

class ICH2DDataset(Dataset):
    """Standard 3-channel dataset (for ablation control)."""
    def __init__(self, df, npy_root, transform, multilabel=True):
        self.df = df.reset_index(drop=True)
        self.npy_root = Path(npy_root)
        self.transform = transform
        self.multilabel = multilabel

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = self.npy_root / f'{row["image_id"]}.npy'
        img = np.load(str(p)).astype(np.uint8) if p.exists() else \
              np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)['image']
        if self.multilabel:
            labels = torch.tensor([row[c] for c in SUBTYPES], dtype=torch.float32)
        else:
            labels = torch.tensor([row['any']], dtype=torch.float32)
        return img, labels


class ICH2_5DDataset(Dataset):
    """2.5D (9-channel) dataset."""
    def __init__(self, df, npy_root, transform, multilabel=True):
        self.df = df.reset_index(drop=True)
        self.npy_root = Path(npy_root)
        self.transform = transform
        self.multilabel = multilabel

    def __len__(self): return len(self.df)

    def _load(self, iid):
        if pd.isna(iid): return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        p = self.npy_root / f'{iid}.npy'
        return np.load(str(p)).astype(np.uint8) if p.exists() else \
               np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.concatenate([
            self._load(row.get('prev_image_id')),
            self._load(row['image_id']),
            self._load(row.get('next_image_id')),
        ], axis=2)
        if self.transform:
            img = self.transform(image=img)['image']
        if self.multilabel:
            labels = torch.tensor([row[c] for c in SUBTYPES], dtype=torch.float32)
        else:
            labels = torch.tensor([row['any']], dtype=torch.float32)
        return img, labels


# ─── Transforms ───────────────────────────────────────────────────────
def get_transforms(n_ch, augment=True):
    mean = MEAN_9CH.tolist() if n_ch == 9 else MEAN_3CH.tolist()
    std  = STD_9CH.tolist()  if n_ch == 9 else STD_3CH.tolist()
    if augment:
        return A.Compose([
            A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.8, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                               rotate_limit=15, p=0.5, border_mode=0),
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
            A.Normalize(mean=mean, std=std, max_pixel_value=255),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Normalize(mean=mean, std=std, max_pixel_value=255), ToTensorV2()])

print('Datasets + transforms ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HELPERS: model builder + mini-train loop  ██
# ══════════════════════════════════════════════════════════════════════════

def build_model(backbone='tf_efficientnet_b4', in_ch=9, n_cls=6, pretrained=True):
    model = timm.create_model(backbone, pretrained=pretrained,
                              num_classes=0, drop_rate=0.3, drop_path_rate=0.2)
    old = model.conv_stem
    new = nn.Conv2d(in_ch, old.out_channels, kernel_size=old.kernel_size,
                    stride=old.stride, padding=old.padding,
                    bias=(old.bias is not None))
    k = in_ch // 3 if in_ch > 3 else 1
    with torch.no_grad():
        if in_ch == 3:
            new.weight.copy_(old.weight)
        else:
            new.weight.copy_(old.weight.repeat(1, k, 1, 1) / k)
        if old.bias is not None:
            new.bias.copy_(old.bias)
    model.conv_stem = new
    model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.num_features, n_cls))
    return model


def run_experiment(name, train_ds, val_ds, model, n_cls, lr=LR, n_ep=ABLATION_EP):
    """Quick training loop for one ablation experiment.
    
    FIXED:
    - Returns dict with best_auc (across all epochs), not just last epoch
    - All variants train for the SAME fixed epoch budget (fair comparison)
    - No early stopping (intentional: fixed budget ensures fair comparison)
    """
    print(f'\n{"─"*55}')
    print(f'  Experiment: {name}')
    print(f'  Train: {len(train_ds):,}  Val: {len(val_ds):,}  '
          f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
    print(f'  Fixed budget: {n_ep} epochs (all variants identical)')
    print(f'{"─"*55}')

    model = model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda')

    # Loss
    if n_cls == 1:
        criterion = nn.BCEWithLogitsLoss()
    else:
        def criterion(logits, targets):
            t = targets * 0.95 + 0.025
            bce = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
            w = torch.ones_like(bce); w[:, 0] = 1.0; w[:, 1:] = 0.4
            return (bce * w).mean()

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                              num_workers=4, pin_memory=True)

    records = []
    best_auc = 0.0
    for ep in range(n_ep):
        model.train()
        ep_loss, n_b = 0.0, 0
        for imgs, labels in tqdm(train_loader, desc=f'  Ep{ep}', leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda'):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            ep_loss += loss.item(); n_b += 1

        # Validate
        model.eval()
        all_p, all_l = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda'):
                    all_p.append(torch.sigmoid(model(imgs)).cpu())
                all_l.append(labels)

        probs  = torch.cat(all_p).numpy()
        labels_np = torch.cat(all_l).numpy()

        if n_cls == 1:
            auc = roc_auc_score(labels_np[:, 0], probs[:, 0])
            auc_str = f'AUC={auc:.4f}'
        else:
            per_cls = [roc_auc_score(labels_np[:, i], probs[:, i])
                       for i in range(min(n_cls, labels_np.shape[1]))
                       if labels_np[:, i].sum() > 0]
            auc = np.mean(per_cls)
            auc_str = f'AUC_mean={auc:.4f} any={per_cls[0]:.4f}'

        best_auc = max(best_auc, auc)
        print(f'  Ep{ep}  loss={ep_loss/n_b:.4f}  {auc_str}  (best={best_auc:.4f})')
        records.append({'epoch': ep, 'loss': ep_loss/n_b, 'auc': auc})

    del model; torch.cuda.empty_cache(); gc.collect()
    return {'records': records, 'best_auc': best_auc, 'last_auc': records[-1]['auc']}

print('Helpers ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LOAD DATA + SUBSET  ██
# ══════════════════════════════════════════════════════════════════════════
manifest = pd.read_csv(MANIFEST_PATH)
train_full = manifest[manifest['fold'] != FOLD_ID]
val_full   = manifest[manifest['fold'] == FOLD_ID]

# Ablation subset (random sample, deterministic)
np.random.seed(SEED)
abl_idx = np.random.choice(len(train_full), size=int(len(train_full)*ABLATION_FRAC),
                           replace=False)
train_abl = train_full.iloc[abl_idx]

print(f'Ablation train: {len(train_abl):,} / {len(train_full):,}')
print(f'Val:            {len(val_full):,}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPERIMENT 1: 2D vs 2.5D  ██
# ══════════════════════════════════════════════════════════════════════════
results_all = {}

# ─── Control: 2D (3ch) ────────────────────────────────────────────────
seed_everything(SEED)
r1a = run_experiment(
    '2D  (3ch, B4)', 
    ICH2DDataset(train_abl, NPY_CACHE_DIR, get_transforms(3, True)),
    ICH2DDataset(val_full,  NPY_CACHE_DIR, get_transforms(3, False)),
    build_model('tf_efficientnet_b4', in_ch=3, n_cls=6), n_cls=6)
results_all['2D (3ch)'] = r1a['best_auc']

# ─── Treatment: 2.5D (9ch) ────────────────────────────────────────────
seed_everything(SEED)
r1b = run_experiment(
    '2.5D (9ch, B4)',
    ICH2_5DDataset(train_abl, NPY_CACHE_DIR, get_transforms(9, True)),
    ICH2_5DDataset(val_full,  NPY_CACHE_DIR, get_transforms(9, False)),
    build_model('tf_efficientnet_b4', in_ch=9, n_cls=6), n_cls=6)
results_all['2.5D (9ch)'] = r1b['best_auc']

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPERIMENT 2: B0 vs B4  ██
# ══════════════════════════════════════════════════════════════════════════
seed_everything(SEED)
r2a = run_experiment(
    'B0 (2.5D)',
    ICH2_5DDataset(train_abl, NPY_CACHE_DIR, get_transforms(9, True)),
    ICH2_5DDataset(val_full,  NPY_CACHE_DIR, get_transforms(9, False)),
    build_model('tf_efficientnet_b0', in_ch=9, n_cls=6), n_cls=6)
results_all['B0 (2.5D)'] = r2a['best_auc']

# B4 already tested in Exp 1b — reuse result (same config, same seed)
results_all['B4 (2.5D)'] = results_all['2.5D (9ch)']
print(f'  B4 reused from Exp1b (same config): AUC={results_all["B4 (2.5D)"]:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPERIMENT 3: Binary vs Multi-label  ██
# ══════════════════════════════════════════════════════════════════════════
# NOTE: Binary model outputs 1 class, multi-label outputs 6.
# Both are compared on the "any" class AUC only.
# The multi-label result is from Exp1b (2.5D B4) — same architecture,
# so the comparison isolates the formulation choice.

seed_everything(SEED)
r3a = run_experiment(
    'Binary (any only, 2.5D B4)',
    ICH2_5DDataset(train_abl, NPY_CACHE_DIR, get_transforms(9, True), multilabel=False),
    ICH2_5DDataset(val_full,  NPY_CACHE_DIR, get_transforms(9, False), multilabel=False),
    build_model('tf_efficientnet_b4', in_ch=9, n_cls=1), n_cls=1)
results_all['Binary'] = r3a['best_auc']
results_all['Multi-label'] = results_all['2.5D (9ch)']
print(f'  NOTE: Binary AUC is "any"-class only; Multi-label AUC is mean of 6 classes.')
print(f'  This comparison is approximate — binary focuses all capacity on one task.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPERIMENT 4: Augmentation level  ██
# ══════════════════════════════════════════════════════════════════════════
seed_everything(SEED)
r4a = run_experiment(
    'Minimal aug (flip only)',
    ICH2_5DDataset(train_abl, NPY_CACHE_DIR,
                   A.Compose([
                       A.HorizontalFlip(p=0.5),
                       A.Normalize(mean=MEAN_9CH.tolist(), std=STD_9CH.tolist(), max_pixel_value=255),
                       ToTensorV2()])),
    ICH2_5DDataset(val_full,  NPY_CACHE_DIR, get_transforms(9, False)),
    build_model('tf_efficientnet_b4', in_ch=9, n_cls=6), n_cls=6)
results_all['Minimal aug'] = r4a['best_auc']
results_all['Full aug'] = results_all['2.5D (9ch)']

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  ABLATION SUMMARY TABLE + CHART  ██
# ══════════════════════════════════════════════════════════════════════════

# Build comparison table
comparisons = [
    ('Input',       '2D (3ch)', results_all['2D (3ch)'],    '2.5D (9ch)', results_all['2.5D (9ch)']),
    ('Backbone',    'B0',       results_all['B0 (2.5D)'],   'B4',         results_all['B4 (2.5D)']),
    ('Formulation', 'Binary',   results_all['Binary'],      'Multi-label',results_all['Multi-label']),
    ('Aug',         'Minimal',  results_all['Minimal aug'], 'Full',       results_all['Full aug']),
]

print(f'\n{"Experiment":<15} {"Control":<18} {"AUC":>6}  {"Treatment":<18} {"AUC":>6}  {"Δ":>7}')
print('─' * 80)
for exp, ctrl_name, ctrl_auc, trt_name, trt_auc in comparisons:
    delta = trt_auc - ctrl_auc
    arrow = '↑' if delta > 0 else '↓' if delta < 0 else '→'
    print(f'{exp:<15} {ctrl_name:<18} {ctrl_auc:.4f}  {trt_name:<18} {trt_auc:.4f}  '
          f'{arrow}{abs(delta):.4f}')

print(f'\n⚠️  Under constrained setting ({ABLATION_FRAC*100:.0f}% subset, '
      f'{ABLATION_EP} epochs, fold {FOLD_ID} only).')
print(f'   AUC values are best-across-epochs (not last epoch).')
print(f'   Deltas indicate directional trends, not definitive conclusions.')

# ─── Per-experiment training curves ─────────────────────────────────
all_runs = {'2D (3ch)': r1a, '2.5D (9ch)': r1b, 'B0': r2a,
            'Binary': r3a, 'Minimal aug': r4a}

fig, ax = plt.subplots(figsize=(8, 4))
for name, result in all_runs.items():
    epochs = [r['epoch'] for r in result['records']]
    aucs   = [r['auc']   for r in result['records']]
    ax.plot(epochs, aucs, 'o-', label=f'{name} (best={result["best_auc"]:.4f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('AUC')
ax.set_title('Ablation Training Curves')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ablation_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Bar chart (FIXED: bars variable was missing) ──────────────────
fig, ax = plt.subplots(figsize=(10, 5))
variants = list(results_all.keys())
aucs     = [results_all[v] for v in variants]
colors   = ['#2196F3' if 'B4' in v or '2.5D' in v or 'Multi' in v or 'Full' in v
            else '#90CAF9' for v in variants]
bars = ax.barh(variants, aucs, color=colors, edgecolor='white')
ax.set_title(f'Ablation Study ({ABLATION_FRAC*100:.0f}% subset, {ABLATION_EP} epochs)\n'
             f'⚠️ Directional trends only — not definitive conclusions')
ax.set_xlim(0.5, 1.0)
for bar, val in zip(bars, aucs):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ablation_chart.png', dpi=150, bbox_inches='tight')
plt.show()

# Save detailed results
pd.DataFrame(comparisons,
             columns=['experiment', 'control', 'control_auc', 'treatment', 'treatment_auc']
).to_csv(OUTPUT_DIR / 'ablation_results.csv', index=False)

print(f'\nResults saved to ablation_results.csv + ablation_chart.png + ablation_curves.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HEALTH CHECK  ██
# ══════════════════════════════════════════════════════════════════════════
errors = []
if not (OUTPUT_DIR / 'ablation_results.csv').exists():
    errors.append('ablation_results.csv missing')
if not (OUTPUT_DIR / 'ablation_chart.png').exists():
    errors.append('ablation_chart.png missing')
if len(results_all) < 6:
    errors.append(f'Only {len(results_all)} experiment results (expected ≥6)')

health = {
    'notebook': '02_ablation', 'status': 'PASS' if not errors else 'FAIL',
    'errors': errors, 'n_experiments': len(results_all),
    'results': {k: round(v, 5) for k, v in results_all.items()},
}
json.dump(health, open(OUTPUT_DIR / 'health_check_nb02.json', 'w'), indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:', errors)
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   {len(results_all)} experiments completed.')